<a href="https://colab.research.google.com/github/xozi/powerpkg/blob/main-py/problems/ee427_hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Homework 4
Consider the two-bus power system shown in the figure above. The system consists of:
A slack bus (Bus 1) with a voltage magnitude of 1 per unit (p.u.) and a zero voltage angle.
A load bus (Bus 2), where active power (P) and reactive power (Q) are specified.
A transmission line connecting Bus 1 and Bus 2, modeled as an impedance of Z=j0.1 p.u.
System Base Power: Sbase=100 MVA
Tasks:

1. Formulate the power flow equations using the Newton-Raphson method for this system.
2. Determine the unknown variables:
    * Voltage magnitude and angle at Bus 2.
3. Solve the power flow problem iteratively using the Newton-Raphson method.
4. Discuss the convergence of the NR method for this simple two-bus system.
5. Validate your results using either PowerWorld or MATPOWER.

I've converted my previous code and Matlab examples to Python for easy debugging during the exam, which includes this homework.
We start by firsting installing some packages, the main one used in numpy currently for linear algebra, matrix operations, and trigonometric functions.


In [1]:
!pip install numpy==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 823.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 21.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.3 which is incompatible.


Defining our intial conditions we have:

In [2]:
import numpy as np
Vsp = np.array([1.0, 1.0])
Angsp = np.array([0.0, 0.0])
Psp = np.array([0, -200e6/100e6])
Qsp = np.array([0, -100e6/100e6])
busamount = 2

Where the Vsp is the intial voltages specified, Angsp is the intial angles specified, Psp is the active power specified, and Qsp is the reactive power specified. This is a two bus system, so busamount is 2.

Defining our G matrix and B matrix we have:

In [3]:
Z = 0.1j
Y=1/(Z)
Ymatrix = np.array([[Y, -Y], [-Y, Y]])
G = np.real(Ymatrix)
B = np.imag(Ymatrix)

For the next part an interation process (Newton-Raphson) for the unknown variables is done, but we first need to define some functions for the Jacobian matrix. We will also need to clean factors from the final mismatch vector and Jacobian matrix as they will include the slack bus, a value we are referencing but not solving for.

In [4]:
def J1solve(Q, Anglediff, i, j):
    if i == j:
        return (-Q[i] - B[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * Vsp[j] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff)))

def J2solve(P, Anglediff, i, j):
    if i == j:
        return (P[i] + G[i,i] * (Vsp[i]**2))
    else:
        return (-Vsp[i] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff)))

def J3solve(P, Anglediff, i, j):
    if i == j:
        return (P[i] - G[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * Vsp[j] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff)))

def J4solve(Q, Anglediff, i, j):
    if i == j:
        return (Q[i] - B[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff)))


def clean_slack(J1, J2, J3, J4, mismatch):
    J1r=J1[1:, 1:]
    J2r=J2[1:, 1:]
    J3r=J3[1:, 1:]
    J4r=J4[1:, 1:]
    J=np.block([[J1r, J2r], [J3r, J4r]])

    mismatch_reduced = np.concatenate([
        mismatch[0][1:],
        mismatch[1][1:]
    ])

    return J, mismatch_reduced


The final Jacobian should be 8x8
where:

$\frac{\partial P}{\partial V}=J_2=[\frac{\partial P_0}{\partial V_0}, \frac{\partial P_0}{\partial V_1}, \frac{\partial P_1}{\partial V_0}, \frac{\partial P_1}{\partial V_1}]$

$\frac{\partial P}{\partial V}=J_2=[\frac{\partial P_0}{\partial V_0}, \frac{\partial P_0}{\partial V_1}, \frac{\partial P_1}{\partial V_0}, \frac{\partial P_1}{\partial V_1}]$

$\frac{\partial Q}{\partial \theta}=J_3=[\frac{\partial Q_0}{\partial \theta_0}, \frac{\partial Q_0}{\partial \theta_1}, \frac{\partial Q_1}{\partial \theta_0}, \frac{\partial Q_1}{\partial \theta_1}]$

$\frac{\partial Q}{\partial V}=J_4=[\frac{\partial Q_0}{\partial V_0}, \frac{\partial Q_0}{\partial V_1}, \frac{\partial Q_1}{\partial V_0}, \frac{\partial Q_1}{\partial V_1}]$

We need to remove all elements except the last one which is related to PQ load or

$\begin{bmatrix}
J_1 \\
J_2 \\
J_3 \\
J_4
\end{bmatrix} = \begin{bmatrix}
0 & 0 & 0 & 1 \\
0 & 0 & 0 & 1 \\
0 & 0 & 0 & 1 \\
0 & 0 & 0 & 1
\end{bmatrix}$

where 0 is slack bus elements and 1 is PQ load elements
Thus, we only need filter the first 3 elements of J1, J2, J3, J4 to get the jacobians related to PQ. Another thing to look out for is the Jacobian derivitive are in V/V, which means multipaction by the previous voltage is needed for the update (this tricked me till I noticed the units).

The Newton-Raphson iteration is as follows:

In [6]:
dP = np.zeros(len(Psp))
dQ = np.zeros(len(Qsp))

for iter in range(50):
    P = np.zeros(len(Psp))
    Q = np.zeros(len(Qsp))
    J1 = np.zeros((len(Psp), len(Psp)))
    J2 = np.zeros((len(Psp), len(Psp)))
    J3 = np.zeros((len(Psp), len(Psp)))
    J4 = np.zeros((len(Psp), len(Psp)))

    for i in range(busamount):
        for j in range(busamount):
            Anglediff = Angsp[i] - Angsp[j]
            P[i] += Vsp[i] * Vsp[j] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff))
            Q[i] += Vsp[i] * Vsp[j] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff))
        for k in range(busamount):
            Anglediff = Angsp[i] - Angsp[k]
            J1[i, k] += J1solve(Q, Anglediff, i, k)
            J2[i, k] += J2solve(P, Anglediff, i, k)
            J3[i, k] += J3solve(P, Anglediff, i, k)
            J4[i, k] += J4solve(Q, Anglediff, i, k)

    dP = Psp - P
    dQ = Qsp - Q

    mismatch = np.array([dP, dQ])

    J, mismatch = clean_slack(J1, J2, J3, J4, mismatch)
    if np.max(np.abs(mismatch)) < 1e-6:
        print(f"Converged in {iter+1} iterations")
        break

    X = np.linalg.solve(J, mismatch)

    n = len(X) // 2
    delta_theta = X[:n]
    delta_V = X[n:]

    for i in range(1, len(Vsp)):
        Vsp[i] += delta_V[i-1] * Vsp[i]
        Angsp[i] += delta_theta[i-1]


Converged in 5 iterations


From my testing this convergence happens with 5 iterations for a tolerance of 1e-6.
As the iteration continues the mismatch or the change in the active and reactive power continues to decrease. The calculated reactive and active power gets closer to the specified reactive and active power, making the delta very small.

We then do linear solve for X, which is a vertical vector  for the delta in the voltage and angle as per this formula:

$\begin{bmatrix} \Delta \theta \\ \Delta V \end{bmatrix} = J^{-1} \cdot \begin{bmatrix} \Delta P \\ \Delta Q \end{bmatrix}$

I wrote the iteration to be applicable in many different systems if there is only one slack bus, defined at index 0 for the specified V, Theta, P, and Q.

TODO:
The only thing I haven't worked on is the Y matrix solver, which from examples I researched can be done through object orientation using node objects. Given a reference structure to each node we can create line objects defined as from and to nodes, making the admittance matrix a summation of the line objects, and all defined possibilities to and from each node object.

The final results now can be printed:

In [7]:
print("Voltage of Load Bus 2: " + str(Vsp[1]) + " pu")
print("Angle of Load Bus 2: " + str(Angsp[1]*180/np.pi) + " deg")

Voltage of Load Bus 2: 0.8553727142892732 pu
Angle of Load Bus 2: -13.521852232864672 deg


To simulate the system we can use the PowerWorld simulator, or the MATPOWER simulator. I decided with PowerWorld for this problem.

Given the following system and the results presented on Bus 2 when we do a solve simulation on the system are:
<img src='https://drive.google.com/uc?id=18Qh5udSDX0yNVRzQmc4SQ3xXd6800gWB'>
<img src='https://drive.google.com/uc?id=1qeQdI3VBkzWOPx1MwTr6ZLz9PFBynTWi'>

These match thoroughly with the results from the Newton-Raphson method.